# 🛡️ Agent Engineering Challenge

**Duration:** ~60 minutes &nbsp;|&nbsp; **Solo build (peer collaboration encouraged)** &nbsp;|&nbsp; **Claude Messages API**

---

## What You're Building

An AI agent that automates RFP (Request for Proposal) responses for a cybersecurity vendor. Your agent will:

1. **Parse** a questionnaire into categorized questions
2. **Retrieve** relevant source material from a knowledge base
3. **Draft** polished, cited answers
4. **Review** answers for cross-question consistency
5. **Export** structured JSON output

The questionnaire in this notebook is what you'll develop and test against. At the end of the session you'll get a **surprise RFP** to point your agent at — so the goal isn't to hardcode the questions below, it's to build something flexible and robust enough to handle questions you haven't seen yet.

This is a solo build, but you're strongly encouraged to bounce ideas off the people around you — sketch architectures together, compare tool designs, debug each other's tool-use loops. Each person ships their own agent.

### Design Principles to Keep in Mind

- **Generality over specificity.** Tool design, prompts, and parsing should not assume the exact questions below.
- **Failure modes matter.** What happens when the KB doesn't have the answer? When a question spans two categories? When the prospect asks something off-script?
- **Consistency is the real quality bar.** Cross-question contradictions are the #1 failure mode of automated RFP responses — your review step is what separates a toy from something deployable.

### Build Something Worth Showing Off

There's no formal demo and no grading. But there's a room full of smart people who'd love to see something interesting. Past Partner Basecamp builds we've loved: RFP deck builders, robust eval harnesses, analysis UIs to visualize answer accuracy, search UIs that plug into any RFP, agents that route by category to specialized subagents. If you ship the baseline early, spend the remaining time building the thing **you** want to show off.

### Suggested Time Allocation

| Phase | Duration | Focus |
|-------|----------|-------|
| Setup & Planning | 5 min | Verify API, sketch architecture, identify your robustness bets |
| Build | 45 min | Agent loop, prompt design, testing on the included questionnaire |
| Surprise RFP | 5–8 min | Point your agent at the unseen RFP — see what holds up |

Extra time? Push harder on robustness, add tooling, or build the show-off feature you've been sitting on.

---

## Part 0: Environment Setup

Run the cells below to install dependencies and verify your API connection.

In [ ]:
# Install dependencies
!pip install anthropic --upgrade -q

print("✓ Dependencies installed")

In [ ]:
import os
import json
from typing import Optional

import anthropic

# === API KEY SETUP ===
ANTHROPIC_API_KEY = ""  # <-- Paste your Anthropic API key here

if not ANTHROPIC_API_KEY:
    raise ValueError(
        "❌ ANTHROPIC_API_KEY not found.\n"
        "Set it via: export ANTHROPIC_API_KEY='your-key-here'\n"
        "Or paste directly in the cell above."
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✓ Anthropic client initialized")

In [ ]:
# Verify API connectivity
try:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=20,
        messages=[{"role": "user", "content": "Say 'ready' and nothing else."}]
    )
    print(f"✓ API connection verified: {response.content[0].text}")
    print(f"  Model: {response.model}")
except Exception as e:
    print(f"❌ API connection failed: {e}")

---

## Part 1: The Brief

### The Situation

**Helios Security** is a mid-market cybersecurity vendor selling endpoint protection, SIEM, and managed detection & response. Their sales team responds to **40+ RFPs per quarter**, each containing 50–200 questions spanning technical architecture, compliance certifications, pricing models, and company background.

Today, each RFP takes a solutions engineer **6–8 hours** to complete. The process is manual: hunt through a Confluence wiki of past proposals, cross-reference product documentation, and copy-paste answers into a spreadsheet — adjusting tone and specificity for each prospect. Answers frequently **contradict each other** across questions because no one reviews the full response holistically.

Helios wants an AI agent that takes in an RFP questionnaire and produces a draft response in minutes — decomposing questions, retrieving relevant source material, synthesizing polished answers, and flagging anything the human needs to review.

**The goal:** Cut first-draft time from 8 hours to under 15 minutes and eliminate cross-answer inconsistencies.

### The Agent Pipeline

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  PARSE   │───▶│ RETRIEVE │───▶│  DRAFT   │───▶│  REVIEW  │───▶│  EXPORT  │
│          │    │          │    │          │    │          │    │          │
│ Break    │    │ Search   │    │ Generate │    │ Check    │    │ Return   │
│ into Qs  │    │ mock KB  │    │ answers  │    │ for      │    │ struct.  │
│ + tag    │    │ via tool │    │ w/ cites │    │ contra-  │    │ JSON     │
│ category │    │          │    │          │    │ dictions │    │          │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

### Illustrative RFP Questions

The kinds of questions Helios sees in real RFPs span multiple categories — some have crisp answers in the KB, some don't, and some span two categories at once. Examples of the shape (these are **not** the exact questions you'll build against — those are in Part 5, and the surprise RFP at the end will be different again):

| ID | Category | Question |
|----|----------|----------|
| Ex1 | Technical | What is your incident response SLA for a confirmed P1 event? How is escalation handled outside business hours? |
| Ex2 | Compliance | Describe your sub-processor management process and how you notify customers of changes. |
| Ex3 | Pricing | What pricing flexibility is available for multi-year commitments, and how are price increases capped at renewal? |
| Ex4 | Company-Info | What is your customer support model? Describe tiers, channels, and average response times. |
| Ex5 | Technical + Compliance | How does your platform support customer-controlled key management, and which regions are FIPS 140-2 validated? |

Some of these are well-covered by the knowledge base. Some aren't. Your agent should handle both gracefully.

---

## Part 2: Mock Knowledge Base (Pre-Built)

Your agent needs data to retrieve from. Below is a pre-built mock knowledge base with realistic Helios Security content. **You can extend this with additional entries**, but the baseline is ready to use.

In [ ]:
# ============================================================
# MOCK KNOWLEDGE BASE
# Pre-populated with realistic Helios Security content.
# Extend with additional entries if you want richer agent behavior.
# ============================================================

KNOWLEDGE_BASE = {
    "threat_detection": {
        "source": "Helios Platform Architecture Doc v4.2",
        "content": (
            "Helios Sentinel uses a multi-layered detection engine combining "
            "signature-based matching, behavioral analysis, and ML-driven anomaly detection. "
            "Data sources include endpoint telemetry (process events, file system changes, "
            "network connections), cloud workload logs (AWS CloudTrail, Azure Activity Log, "
            "GCP Audit Log), network flow data (NetFlow v9/IPFIX), and email gateway events. "
            "Average detection-to-alert latency is 2.3 seconds for signature matches and "
            "18 seconds for behavioral detections. Our SIEM correlation engine processes "
            "up to 50,000 events per second per tenant."
        ),
        "tags": ["technical", "detection", "latency", "architecture"]
    },
    "compliance_certs": {
        "source": "Helios Compliance & Certifications Register 2025",
        "content": (
            "Current certifications: SOC 2 Type II (audited December 2024 by Deloitte), "
            "ISO 27001:2022 (certified March 2024 by BSI), FedRAMP Moderate (authorized "
            "June 2024, sponsored by DHS), HIPAA (BAA available, last assessment October 2024), "
            "PCI DSS v4.0 Level 1 Service Provider (validated September 2024 by Coalfire). "
            "StateRAMP authorized (January 2025). All certifications maintained on continuous "
            "monitoring basis with quarterly internal audits."
        ),
        "tags": ["compliance", "certifications", "audit", "soc2", "fedramp"]
    },
    "pricing_model": {
        "source": "Helios Commercial Pricing Sheet Q1 2025",
        "content": (
            "Endpoint Protection Platform (EPP+EDR bundle): "
            "500 endpoints: $18/seat/month ($108,000/year). "
            "1,000 endpoints: $15/seat/month ($180,000/year) — 17% volume discount. "
            "5,000 endpoints: $11/seat/month ($660,000/year) — 39% volume discount. "
            "Minimum contract term: 12 months. Multi-year discounts: 2-year = additional 5%, "
            "3-year = additional 10%. SIEM add-on: +$6/seat/month. "
            "MDR add-on: +$12/seat/month. All pricing excludes professional services."
        ),
        "tags": ["pricing", "commercial", "discount", "contract"]
    },
    "financial_services_customers": {
        "source": "Helios Customer Success — Vertical Report 2024",
        "content": (
            "Helios currently serves 47 customers in financial services, including "
            "12 banks, 8 insurance carriers, 15 asset management firms, and 12 fintech companies. "
            "Reference accounts (approved for external use): "
            "1) Meridian National Bank — 3,200 endpoints, EPP+EDR+SIEM, deployed since 2022. "
            "2) Crestview Capital Partners — 850 endpoints, EPP+MDR, deployed since 2023. "
            "3) Apex Insurance Group — 5,100 endpoints, full platform, deployed since 2021. "
            "Average NPS in financial services vertical: 72."
        ),
        "tags": ["company-info", "customers", "financial-services", "references"]
    },
    "data_residency_eu": {
        "source": "Helios Data Sovereignty & Privacy Whitepaper v3.1",
        "content": (
            "Helios supports full EU data residency through dedicated infrastructure in "
            "Frankfurt (AWS eu-central-1) and Dublin (AWS eu-west-1). Customer data never "
            "leaves the selected region. Encryption at rest: AES-256-GCM with customer-managed "
            "keys (AWS KMS or BYOK). Encryption in transit: TLS 1.3 for all API and agent "
            "communications, with certificate pinning for endpoint agents. "
            "GDPR Data Processing Agreement (DPA) included in all EU contracts. "
            "Annual third-party penetration testing by NCC Group. "
            "Data retention: configurable per tenant, default 90 days for raw telemetry, "
            "13 months for aggregated alerts."
        ),
        "tags": ["technical", "compliance", "data-residency", "eu", "encryption", "gdpr"]
    },
    "past_rfp_detection_answer": {
        "source": "Acme Corp RFP Response — March 2024",
        "content": (
            "Q: Describe your real-time threat detection capabilities. "
            "A: Helios Sentinel provides sub-3-second detection for known threat patterns "
            "and under 20 seconds for behavioral anomalies. Our detection engine ingests "
            "endpoint telemetry, network flows, cloud audit logs, and email events. "
            "The SIEM correlation engine handles 50K EPS per tenant. "
            "We maintain a 99.7% true positive rate on our top 100 detection rules, "
            "validated quarterly against MITRE ATT&CK framework."
        ),
        "tags": ["technical", "detection", "past-rfp"]
    },
    "past_rfp_compliance_answer": {
        "source": "NovaTech RFP Response — July 2024",
        "content": (
            "Q: What compliance certifications do you hold? "
            "A: Helios holds SOC 2 Type II, ISO 27001, FedRAMP Moderate, PCI DSS v4.0, "
            "and HIPAA compliance. All certifications are actively maintained with "
            "continuous monitoring. We provide audit reports upon request under NDA. "
            "Our security team of 14 full-time engineers manages compliance programs."
        ),
        "tags": ["compliance", "certifications", "past-rfp"]
    }
}


def search_knowledge_base(query: str, category: Optional[str] = None) -> list[dict]:
    """
    Search the mock knowledge base.
    Returns matching entries ranked by keyword overlap.
    """
    query_terms = set(query.lower().split())
    results = []

    for entry_id, entry in KNOWLEDGE_BASE.items():
        # Score by keyword overlap with content + tags
        entry_text = (entry["content"] + " " + " ".join(entry["tags"])).lower()
        overlap = len(query_terms & set(entry_text.split()))

        # Boost if category matches a tag
        if category and category.lower() in [t.lower() for t in entry["tags"]]:
            overlap += 5

        if overlap > 0:
            results.append({
                "id": entry_id,
                "source": entry["source"],
                "content": entry["content"],
                "relevance_score": overlap
            })

    # Sort by relevance, return top 3
    results.sort(key=lambda x: x["relevance_score"], reverse=True)
    return results[:3]


# Quick test
test_results = search_knowledge_base("threat detection latency", category="technical")
print(f"✓ Knowledge base loaded ({len(KNOWLEDGE_BASE)} entries)")
print(f"  Test search 'threat detection latency': {len(test_results)} results")
print(f"  Top result: {test_results[0]['source']}")

---

## Part 3: Tool Definition

Define the `search_kb` tool so Claude can retrieve information from the knowledge base during its agent loop.

In [ ]:
# Tool definition for the Claude API
SEARCH_KB_TOOL = {
    "name": "search_kb",
    "description": (
        "Search the Helios Security knowledge base for information relevant to "
        "answering an RFP question. Returns up to 3 matching documents with source "
        "attribution. Use this to find product docs, past proposal answers, compliance "
        "records, and pricing information."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query — use keywords from the RFP question"
            },
            "category": {
                "type": "string",
                "enum": ["technical", "compliance", "pricing", "company-info"],
                "description": "Optional category filter to narrow results"
            }
        },
        "required": ["query"]
    }
}


def handle_tool_call(tool_name: str, tool_input: dict) -> str:
    """Execute a tool call and return the result as a string."""
    if tool_name == "search_kb":
        results = search_knowledge_base(
            query=tool_input["query"],
            category=tool_input.get("category")
        )
        return json.dumps(results, indent=2)
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})


print("✓ Tool definition ready")

---

## Part 4: Level 0 Agent (Working Baseline)

Below is a working agent that handles a **single question** end-to-end. It:
- Sends the question to Claude with the `search_kb` tool available
- Handles the tool use loop (Claude calls the tool → we execute it → send results back)
- Returns a structured answer

**Run this to confirm the agent loop works**, then extend it in Part 5.

In [ ]:
SYSTEM_PROMPT = """You are an AI assistant helping Helios Security respond to RFP questionnaires.

For each question, you must:
1. Use the search_kb tool to find relevant source material
2. Draft a professional, detailed answer grounded in the retrieved sources
3. Cite your sources by name
4. If the knowledge base doesn't contain enough information, flag the answer as low-confidence

Return your answer as JSON with this structure:
{
    "question_id": "Q1",
    "category": "technical",
    "answer": "Your drafted answer here...",
    "sources": ["Source Name 1", "Source Name 2"],
    "confidence": "high" | "medium" | "low",
    "flags": ["any concerns or notes for human review"]
}

Be specific, professional, and concise. Use concrete numbers from the source material."""


def answer_single_question(
    question_id: str,
    question_text: str,
    category: str,
    model: str = "claude-sonnet-4-20250514",
) -> dict:
    """
    Level 0 agent: answers a single RFP question with tool use.
    Handles the full tool use loop.
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Answer this RFP question.\n\n"
                f"Question ID: {question_id}\n"
                f"Category: {category}\n"
                f"Question: {question_text}\n\n"
                f"Search the knowledge base for relevant information, then draft your answer."
            )
        }
    ]

    # Agent loop: keep going until Claude stops calling tools
    max_turns = 5  # Safety limit
    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=messages,
            tools=[SEARCH_KB_TOOL],
        )

        # If Claude is done (no more tool calls), extract the answer
        if response.stop_reason == "end_turn":
            # Find the text block with the JSON answer
            for block in response.content:
                if block.type == "text":
                    try:
                        # Try to parse JSON from the response
                        text = block.text
                        # Handle markdown code blocks
                        if "```json" in text:
                            text = text.split("```json")[1].split("```")[0]
                        elif "```" in text:
                            text = text.split("```")[1].split("```")[0]
                        return json.loads(text.strip())
                    except json.JSONDecodeError:
                        return {"raw_response": block.text, "parse_error": True}
            break

        # If Claude wants to use a tool, execute it and continue
        if response.stop_reason == "tool_use":
            # Add assistant's response to message history
            messages.append({"role": "assistant", "content": response.content})

            # Execute each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = handle_tool_call(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            messages.append({"role": "user", "content": tool_results})

    return {"error": "Max turns reached without completing"}


print("✓ Level 0 agent defined")

In [ ]:
# Test: Answer Q1
print("Answering Q1 (threat detection)...")
result = answer_single_question(
    question_id="Q1",
    question_text=(
        "Describe your platform's approach to real-time threat detection. "
        "What data sources are ingested, and what is the average detection-to-alert latency?"
    ),
    category="technical",
)

print(json.dumps(result, indent=2))

---

## Part 5: Multi-Agent Architecture

The Level 0 agent (Part 4) handles one question end-to-end. To produce a deployable
RFP responder we split the work across **five specialist agents**, each with a narrow
contract, explicit JSON output, and a model chosen for its cost/capability sweet spot.

| # | Agent | Model | Job |
|---|-------|-------|-----|
| 1 | **Parser** | `claude-haiku-4-5` | RFP text -> list of `{id, category, text}` |
| 2 | **Retriever** | `claude-haiku-4-5` | Per-question tool loop over `search_kb` |
| 3 | **Drafter** | `claude-sonnet-4-6` | Per-question answer JSON with citations + confidence |
| 4 | **Validator** | `claude-sonnet-4-6` | Single pass across all drafts -> issue list |
| 5 | **Reviser** | `claude-sonnet-4-6` | Re-draft any flagged answer with critique |

**Why the Haiku/Sonnet split.** Parser and Retriever are high-volume, low-judgment
roles (extract structure, drive a search loop). Haiku handles those at ~4x lower
cost. Drafter / Validator / Reviser produce or judge customer-facing prose -- that
is where Sonnet's quality bar pays for itself.

### The six failure modes we budget for

1. **Malformed RFP** -- Parser falls back, tags `parse_quality: low`, run continues.
2. **KB miss** -- Retriever sets `kb_gap: true`; Drafter answers narratively, never invents numbers.
3. **API error mid-stage** -- caught per-question, recorded as a flag, other questions continue.
4. **Bad JSON from any agent** -- `safe_parse_json` 3-tier fallback (parse -> regex extract -> raw capture).
5. **Validator contradiction** -- bounded revision (default 1 pass); if still flagged, ship with `unresolved-contradiction`.
6. **API key missing** -- already enforced in Part 0; agents never half-start.

### Pipeline flow

```
            +----------+
RFP text -> |  PARSER  | -> questions[]
            +----------+
                  |
       per question (parallel safe)
                  v
            +-----------+
            | RETRIEVER | -> hits[], kb_gap
            +-----------+
                  |
                  v
            +-----------+
            |  DRAFTER  | -> draft answer JSON
            +-----------+
                  |
                  v
            +-----------+        flagged?
all drafts->| VALIDATOR | --------> +---------+
            +-----------+    yes    | REVISER | -> revised draft
                  |                 +---------+
                  | no flags                |
                  v <----------------------+
            final answers + scoreboard
```


In [ ]:
# Model selection for the agent team.
# Sonnet handles drafting and judgment; Haiku handles parsing and search loops.
PARSER_MODEL = "claude-haiku-4-5"
RETRIEVER_MODEL = "claude-haiku-4-5"
DRAFTER_MODEL = "claude-sonnet-4-6"
VALIDATOR_MODEL = "claude-sonnet-4-6"
REVISER_MODEL = "claude-sonnet-4-6"

print(f"Parser/Retriever: {PARSER_MODEL}")
print(f"Drafter/Validator/Reviser: {DRAFTER_MODEL}")


In [ ]:
# ============================================================
# System prompts -- one per agent. Each demands strict JSON.
# ============================================================

PARSER_PROMPT = """You are the PARSER agent in an RFP response pipeline.

Your job: given the raw text of an RFP, extract every question and tag each with
a category. Categories are exactly: technical | compliance | pricing | company-info | other.

Return STRICT JSON with this schema -- nothing else, no prose, no code fences:

{
  "questions": [
    {"id": "Q1", "category": "technical", "text": "..."},
    {"id": "Q2", "category": "compliance", "text": "..."}
  ],
  "parse_quality": "high"
}

Rules:
- IDs are sequential: Q1, Q2, Q3, ...
- If the RFP is malformed or you cannot reliably identify discrete questions,
  do your best blank-line / sentence-based chunking and set "parse_quality": "low".
- Never invent questions. If only 3 are present, return 3.
- Strip numbering ("1.", "Q1:", "a)") from the text field.
"""

RETRIEVER_PROMPT = """You are the RETRIEVER agent.

Your job: given a single RFP question, use the search_kb tool to find the most
relevant supporting material. Make 1-3 focused searches; do not exceed 4 tool
turns. Then return STRICT JSON -- no prose, no fences:

{
  "hits": [
    {"source": "<source name>", "content": "<verbatim excerpt>", "relevance_score": <int>}
  ],
  "kb_gap": false
}

Rules:
- If your searches return nothing useful, set "kb_gap": true and "hits": [].
- Do NOT draft an answer. Only retrieve.
- Prefer specificity in your queries -- include numbers, product names, certification names.
"""

DRAFTER_PROMPT = """You are the DRAFTER agent.

Your job: write a professional, cited answer to one RFP question using ONLY the
retrieved hits. Return STRICT JSON -- no prose, no fences:

{
  "question_id": "Q1",
  "category": "technical",
  "answer": "...",
  "sources": ["Source Name 1"],
  "confidence": "high",
  "flags": []
}

Rules:
- confidence is one of: high, medium, low.
- If kb_gap is true: confidence MUST be "low", flags MUST include "kb-gap",
  and the answer must narratively acknowledge the gap. Do NOT invent numbers,
  dates, dollar amounts, or certification names that are not in the hits.
- Every numeric or factual claim must be traceable to a hit. List the source
  names you actually drew from in "sources".
- Tone: professional, concise, vendor-appropriate. No marketing fluff.
"""

VALIDATOR_PROMPT = """You are the VALIDATOR agent.

You receive ALL drafted answers for an RFP. Your job is to find cross-answer
problems. Return STRICT JSON -- no prose, no fences:

{
  "issues": [
    {"question_id": "Q2", "issue_type": "contradiction", "detail": "..."}
  ]
}

issue_type is one of:
- "contradiction"     -- two answers state facts that cannot both be true
                        (different dates, prices, certs, capacities, etc.)
- "tone-drift"        -- one answer is markedly more/less formal than the others
- "unsupported-claim" -- a specific number, date, or claim with no source backing

Rules:
- Empty issues list is valid and expected when drafts are consistent.
- Reference the question_id of the answer that should be revised (the weaker one,
  or the one whose claim is less defensible).
- Be precise in "detail" -- quote the conflicting fragments.
"""

REVISER_PROMPT = """You are the REVISER agent.

You receive: one RFP question, its retrieval hits, the original draft, and a
validator critique. Your job: produce a revised draft that resolves the critique
while staying grounded in the hits.

Return STRICT JSON in the same schema as the DRAFTER -- no prose, no fences:

{
  "question_id": "...",
  "category": "...",
  "answer": "...",
  "sources": [...],
  "confidence": "high|medium|low",
  "flags": [...]
}

Rules:
- Address the critique explicitly. If a contradiction was raised, reconcile or
  hedge with a source citation.
- If the critique cannot be resolved from the hits, keep the answer but add
  "unresolved-contradiction" to flags and lower confidence one notch.
- Never invent facts not in hits.
"""

print("Agent system prompts loaded:",
      "PARSER, RETRIEVER, DRAFTER, VALIDATOR, REVISER")


In [ ]:
import re

def safe_parse_json(text: str) -> dict:
    """
    3-tier resilient JSON extraction.

    Tier 1: direct json.loads of the stripped text.
    Tier 2: regex-extract the first {...} block and parse it.
    Tier 3: give up; return the raw text with parse_error=True.

    Every agent call routes through this so malformed model output never
    crashes the pipeline.
    """
    if text is None:
        return {"raw": "", "parse_error": True}

    s = text.strip()
    # Strip common code fences if present.
    if s.startswith("```"):
        # remove leading fence (possibly ```json)
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```\s*$", "", s)

    # Tier 1
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass

    # Tier 2: greedy {...} extract
    match = re.search(r"\{[\s\S]*\}", s)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    # Tier 3
    return {"raw": text, "parse_error": True}


print("safe_parse_json ready")


In [ ]:
import re

def _extract_text(response) -> str:
    """Concatenate all text blocks from an Anthropic response."""
    return "".join(b.text for b in response.content if getattr(b, "type", None) == "text")


def run_parser(rfp_text: str) -> dict:
    """
    PARSER agent: RFP text -> {questions: [...], parse_quality: high|low}.
    Uses Haiku. Never raises -- on any failure returns parse_quality=low.
    """
    try:
        response = client.messages.create(
            model=PARSER_MODEL,
            max_tokens=2048,
            system=PARSER_PROMPT,
            messages=[{"role": "user", "content": f"RFP TEXT:\n\n{rfp_text}"}],
        )
        parsed = safe_parse_json(_extract_text(response))
    except Exception as e:
        return {"questions": [], "parse_quality": "low", "error": str(e)}

    if parsed.get("parse_error") or not isinstance(parsed.get("questions"), list):
        # Fallback: blank-line chunking. Best-effort, marks parse_quality=low.
        chunks = [chunk.strip() for chunk in re.split(r"\n\s*\n", rfp_text) if chunk.strip()]
        chunks = [c for c in chunks if len(c) > 20]
        fallback_qs = [
            {"id": f"Q{i}", "category": "other", "text": chunk}
            for i, chunk in enumerate(chunks, start=1)
        ]
        return {"questions": fallback_qs, "parse_quality": "low", "raw": parsed.get("raw", "")}

    # Defensive cleanup: ensure each question has id/category/text strings.
    cleaned = []
    for i, q in enumerate(parsed["questions"], start=1):
        if not isinstance(q, dict):
            continue
        cleaned.append({
            "id": str(q.get("id") or f"Q{i}"),
            "category": str(q.get("category") or "other"),
            "text": str(q.get("text") or "").strip(),
        })
    return {"questions": cleaned,
            "parse_quality": parsed.get("parse_quality", "high")}


print("run_parser ready")


In [ ]:
def run_retriever(question: dict, max_tool_turns: int = 4) -> dict:
    """
    RETRIEVER agent: per-question tool loop. Returns {hits, kb_gap}.
    Uses Haiku + the existing SEARCH_KB_TOOL / handle_tool_call from Part 3.
    """
    user_msg = (
        f"Find supporting material for this RFP question.\n\n"
        f"ID: {question.get('id')}\n"
        f"Category: {question.get('category')}\n"
        f"Question: {question.get('text')}\n\n"
        f"Run 1-3 focused searches, then return the hits JSON."
    )
    messages = [{"role": "user", "content": user_msg}]

    try:
        for _ in range(max_tool_turns):
            response = client.messages.create(
                model=RETRIEVER_MODEL,
                max_tokens=1500,
                system=RETRIEVER_PROMPT,
                messages=messages,
                tools=[SEARCH_KB_TOOL],
            )
            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        result = handle_tool_call(block.name, block.input)
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        })
                messages.append({"role": "user", "content": tool_results})
                continue
            # end_turn (or other) -- parse final JSON
            parsed = safe_parse_json(_extract_text(response))
            if parsed.get("parse_error"):
                return {"hits": [], "kb_gap": True, "parse_error": True}
            hits = parsed.get("hits") or []
            kb_gap = bool(parsed.get("kb_gap")) or len(hits) == 0
            return {"hits": hits, "kb_gap": kb_gap}
        # exceeded tool turns
        return {"hits": [], "kb_gap": True, "flags": ["retriever-tool-turn-limit"]}
    except Exception as e:
        return {"hits": [], "kb_gap": True, "error": str(e)}


print("run_retriever ready")


In [ ]:
def run_drafter(question: dict, hits: list, kb_gap: bool) -> dict:
    """
    DRAFTER agent: produce the answer JSON for one question.
    Uses Sonnet. Generalization of answer_single_question (Part 4) with
    explicit context injection rather than letting the model drive search.
    """
    payload = {
        "question_id": question.get("id"),
        "category": question.get("category"),
        "question_text": question.get("text"),
        "kb_gap": kb_gap,
        "hits": hits,
    }
    user_msg = (
        "Draft the answer for this question using ONLY the provided hits.\n\n"
        + json.dumps(payload, indent=2)
    )

    try:
        response = client.messages.create(
            model=DRAFTER_MODEL,
            max_tokens=2048,
            system=DRAFTER_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        parsed = safe_parse_json(_extract_text(response))
    except Exception as e:
        return {
            "question_id": question.get("id"),
            "category": question.get("category"),
            "answer": "",
            "sources": [],
            "confidence": "low",
            "flags": ["drafter-error", str(e)],
        }

    if parsed.get("parse_error"):
        return {
            "question_id": question.get("id"),
            "category": question.get("category"),
            "answer": parsed.get("raw", ""),
            "sources": [],
            "confidence": "low",
            "flags": ["drafter-parse-error"],
        }

    # Normalize / enforce contracts.
    parsed.setdefault("question_id", question.get("id"))
    parsed.setdefault("category", question.get("category"))
    parsed.setdefault("answer", "")
    parsed.setdefault("sources", [])
    parsed.setdefault("confidence", "low")
    parsed.setdefault("flags", [])
    if kb_gap:
        parsed["confidence"] = "low"
        if "kb-gap" not in parsed["flags"]:
            parsed["flags"].append("kb-gap")
    return parsed


print("run_drafter ready")


In [ ]:
def run_validator(drafts: list) -> dict:
    """
    VALIDATOR agent: single Sonnet call across ALL drafts.
    Returns {issues: [{question_id, issue_type, detail}]}.
    """
    user_msg = (
        "Review these drafted answers for cross-answer issues.\n\n"
        + json.dumps(drafts, indent=2)
    )
    try:
        response = client.messages.create(
            model=VALIDATOR_MODEL,
            max_tokens=2048,
            system=VALIDATOR_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        parsed = safe_parse_json(_extract_text(response))
    except Exception as e:
        return {"issues": [], "error": str(e)}

    if parsed.get("parse_error") or not isinstance(parsed.get("issues"), list):
        return {"issues": []}
    return {"issues": parsed["issues"]}


print("run_validator ready")


In [ ]:
def run_reviser(question: dict, hits: list, original_draft: dict, critique: dict) -> dict:
    """
    REVISER agent: re-draft a single flagged answer given the validator critique.
    Uses Sonnet. Returns the updated draft (same schema as run_drafter).
    """
    payload = {
        "question": question,
        "hits": hits,
        "original_draft": original_draft,
        "critique": critique,
    }
    user_msg = (
        "Revise the draft to resolve the critique. Stay grounded in hits.\n\n"
        + json.dumps(payload, indent=2)
    )
    try:
        response = client.messages.create(
            model=REVISER_MODEL,
            max_tokens=2048,
            system=REVISER_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
        )
        parsed = safe_parse_json(_extract_text(response))
    except Exception as e:
        revised = dict(original_draft)
        revised.setdefault("flags", []).append(f"reviser-error: {e}")
        return revised

    if parsed.get("parse_error"):
        revised = dict(original_draft)
        revised.setdefault("flags", []).append("reviser-parse-error")
        return revised

    parsed.setdefault("question_id", original_draft.get("question_id"))
    parsed.setdefault("category", original_draft.get("category"))
    parsed.setdefault("sources", original_draft.get("sources", []))
    parsed.setdefault("confidence", original_draft.get("confidence", "low"))
    parsed.setdefault("flags", [])
    return parsed


print("run_reviser ready")


---

## Part 6: Pipeline Orchestration (bounded revision loop)

`process_rfp` wires the five agents together with try/except around each stage so
a single bad question can never sink the whole run. Failures are captured as
flags in the result rather than raised.

- `max_revisions` defaults to 1 (the spec's failure budget).
- Per-question retrieval failures still produce a (low-confidence) draft.
- The validator runs once over all drafts; any flagged question goes through one
  reviser pass.
- A scoreboard summarizes the run at a glance.


In [ ]:
def process_rfp(rfp_text: str, max_revisions: int = 1, rfp_name: str = "unnamed") -> dict:
    """
    Drive the full multi-agent pipeline.
    Never raises -- every stage failure is captured in the result.
    """
    scoreboard = {
        "parsed": 0,
        "flagged": 0,
        "revised": 0,
        "kb_gaps": 0,
        "parse_errors": 0,
    }

    # ---- Parse ----
    try:
        parse_result = run_parser(rfp_text)
    except Exception as e:
        parse_result = {"questions": [], "parse_quality": "low", "error": str(e)}

    questions = parse_result.get("questions", [])
    scoreboard["parsed"] = len(questions)
    if parse_result.get("parse_quality") == "low":
        scoreboard["parse_errors"] += 1

    # ---- Retrieve + Draft per question ----
    per_q_hits = {}
    drafts = []
    for q in questions:
        try:
            retrieval = run_retriever(q)
        except Exception as e:
            retrieval = {"hits": [], "kb_gap": True, "error": str(e)}
        per_q_hits[q["id"]] = retrieval.get("hits", [])
        if retrieval.get("kb_gap"):
            scoreboard["kb_gaps"] += 1

        try:
            draft = run_drafter(q, retrieval.get("hits", []), retrieval.get("kb_gap", False))
        except Exception as e:
            draft = {
                "question_id": q["id"], "category": q["category"],
                "answer": "", "sources": [], "confidence": "low",
                "flags": ["drafter-exception", str(e)],
            }
        drafts.append(draft)

    # ---- Validate ----
    try:
        validation = run_validator(drafts) if drafts else {"issues": []}
    except Exception as e:
        validation = {"issues": [], "error": str(e)}
    issues = validation.get("issues", [])
    flagged_ids = {issue.get("question_id") for issue in issues if issue.get("question_id")}
    scoreboard["flagged"] = len(flagged_ids)

    # ---- Bounded revision loop ----
    revisions = []
    if flagged_ids and max_revisions > 0:
        q_lookup = {q["id"]: q for q in questions}
        draft_lookup = {d.get("question_id"): d for d in drafts}
        for qid in flagged_ids:
            q = q_lookup.get(qid)
            orig = draft_lookup.get(qid)
            if not q or not orig:
                continue
            critique = {"issues": [i for i in issues if i.get("question_id") == qid]}
            try:
                revised = run_reviser(q, per_q_hits.get(qid, []), orig, critique)
            except Exception as e:
                revised = dict(orig)
                revised.setdefault("flags", []).append(f"reviser-exception: {e}")
            # replace in drafts list
            for idx, d in enumerate(drafts):
                if d.get("question_id") == qid:
                    drafts[idx] = revised
                    break
            revisions.append({"question_id": qid, "critique": critique})
            scoreboard["revised"] += 1

    # ---- Post-revision recheck ----
    # Re-run validator once after the bounded revision pass. Any issue that
    # survives gets an unresolved-contradiction flag on the affected draft so
    # downstream consumers know the contradiction shipped.
    if revisions:
        try:
            recheck = run_validator(drafts) if drafts else {"issues": []}
        except Exception as e:
            recheck = {"issues": [], "error": str(e)}
        validation = recheck
        draft_lookup = {d.get("question_id"): d for d in drafts}
        for issue in recheck.get("issues", []):
            qid = issue.get("question_id")
            d = draft_lookup.get(qid)
            if d and "unresolved-contradiction" not in d.get("flags", []):
                d.setdefault("flags", []).append("unresolved-contradiction")

    # Count parse_errors across drafts too
    for d in drafts:
        if any("parse-error" in str(f) for f in d.get("flags", [])):
            scoreboard["parse_errors"] += 1

    return {
        "rfp_name": rfp_name,
        "questions": questions,
        "answers": drafts,
        "validation": validation,
        "revisions": revisions,
        "scoreboard": scoreboard,
        "parse_quality": parse_result.get("parse_quality", "high"),
    }


print("process_rfp ready")


In [ ]:
# Run the pipeline against the baseline RFP_QUESTIONS (rendered as a markdown table).
def _baseline_rfp_text(questions: list[dict]) -> str:
    lines = ["| ID | Category | Question |", "|----|----------|----------|"]
    for q in questions:
        text = q["text"].replace("|", "/")
        lines.append(f"| {q['id']} | {q['category']} | {text} |")
    return "\n".join(lines)


BASELINE_RFP_TEXT = _baseline_rfp_text(RFP_QUESTIONS)

print("Running multi-agent pipeline on baseline RFP...\n")
baseline_result = process_rfp(BASELINE_RFP_TEXT, max_revisions=1, rfp_name="baseline")

print("Scoreboard:")
print(json.dumps(baseline_result["scoreboard"], indent=2))
print(f"\nparse_quality: {baseline_result['parse_quality']}")
print(f"answers: {len(baseline_result['answers'])}")
print(f"validator issues: {len(baseline_result['validation'].get('issues', []))}")
print(f"revisions: {len(baseline_result['revisions'])}")


---

## Part 7: Synthetic RFP Fixtures

Five fixtures, each stressing a different failure mode. These are deterministic
fixtures (not generated at runtime) so demos are repeatable.

| Fixture | What it tests |
|---------|---------------|
| `clean_table` | Happy path -- structured markdown table, same 5 baseline questions |
| `numbered_list` | Parser robustness on `1. ... 2. ...` formatting |
| `prose` | Parser robustness on questions buried in narrative prose |
| `edge_cases` | KB gap handling, off-category routing, ambiguity |
| `adversarial` | Validator contradiction detection (same fact asked two ways) |


In [ ]:
SYNTHETIC_RFPS = {
    "clean_table": (
        "| ID | Category | Question |\n"
        "|----|----------|----------|\n"
        "| Q1 | technical | Describe your platform's approach to real-time threat detection, "
        "including data sources ingested and average detection-to-alert latency. |\n"
        "| Q2 | compliance | List all compliance certifications your organization currently "
        "holds and the date of most recent audit for each. |\n"
        "| Q3 | pricing | Provide per-seat pricing for 500, 1,000, and 5,000 endpoints. "
        "Are volume discounts available? Minimum contract term? |\n"
        "| Q4 | company-info | How many customers do you currently serve in the financial "
        "services vertical? Provide 2-3 reference accounts. |\n"
        "| Q5 | technical | How does your platform handle EU data residency? Describe "
        "encryption at rest and in transit. |\n"
    ),

    "numbered_list": (
        "RFP Questionnaire -- Section 3: Vendor Technical & Commercial\n\n"
        "1. What is your detection-to-alert latency for known signatures versus behavioral anomalies?\n"
        "2. Which compliance certifications do you currently hold, and when were they last audited?\n"
        "3. What is your per-seat pricing for a 1,000-endpoint deployment? Are multi-year discounts available?\n"
        "4. Provide reference accounts in the financial services industry.\n"
        "5. Describe EU data residency support including encryption in transit.\n"
        "6. What is the maximum events-per-second your SIEM correlation engine can process per tenant?\n"
    ),

    "prose": (
        "We are evaluating endpoint protection vendors for our 2025 refresh. Our security "
        "team is most interested in understanding how your platform detects threats in "
        "real time, particularly what telemetry sources you ingest. Equally important to "
        "our legal team is the question of which compliance certifications you maintain "
        "today and how recently they were audited. We will be deploying to roughly 1,000 "
        "endpoints, so we would also like a clear sense of your per-seat pricing at that "
        "scale and whether multi-year commitments unlock any discount. Finally, since a "
        "large portion of our workforce operates in the EU, we need to understand your "
        "data residency model and the encryption posture for data both at rest and in transit."
    ),

    "edge_cases": (
        "1. Who founded Helios Security and in what year?\n"
        "2. What is the current tenure of your Chief Executive Officer?\n"
        "3. Could you describe, in general terms, your approach to handling things?\n"
        "4. What is your detection-to-alert latency for behavioral anomalies?\n"
        "5. How many financial-services customers do you currently serve?\n"
    ),

    "adversarial": (
        "1. When was your organization granted FedRAMP Moderate authorization?\n"
        "2. Please confirm the year in which Helios first received FedRAMP authorization "
        "and identify the sponsoring agency.\n"
        "3. What is the per-seat monthly price for a 500-endpoint deployment?\n"
        "4. For a deployment of a few hundred seats, roughly what should we budget "
        "per seat per month for the EPP and EDR bundle?\n"
        "5. What is your SIEM correlation engine's per-tenant events-per-second ceiling?\n"
    ),
}

print(f"Loaded {len(SYNTHETIC_RFPS)} synthetic RFP fixtures: {list(SYNTHETIC_RFPS.keys())}")


---

## Part 8: Resilient JSON Extraction

Every agent is instructed to return strict JSON, but models drift -- they wrap
output in ```json fences, add a sentence of prose, or close with a trailing
comma. `safe_parse_json` (defined in Part 5) gives us a 3-tier fallback:

1. **Tier 1 -- `json.loads`** on the stripped text (after stripping common code fences).
2. **Tier 2 -- regex `\{[\s\S]*\}`** to lift the first JSON object out of surrounding prose.
3. **Tier 3 -- raw capture** with `parse_error: True` so the pipeline can flag and continue.

The stress test below confirms each tier behaves correctly.


In [ ]:
# Stress-test safe_parse_json on four shapes.
samples = {
    "clean":          '{"a": 1, "b": "hello"}',
    "fenced":         '```json\n{"a": 1, "b": "hello"}\n```',
    "in_prose":       'Sure! Here is your JSON:\n{"a": 1, "b": "hello"}\nLet me know if you want changes.',
    "garbage":        'I cannot return JSON for that question.',
}

for name, text in samples.items():
    parsed = safe_parse_json(text)
    print(f"--- {name} ---")
    print(parsed)

# Assertions for each tier
assert safe_parse_json(samples["clean"]) == {"a": 1, "b": "hello"}, "Tier 1 failed"
assert safe_parse_json(samples["fenced"]) == {"a": 1, "b": "hello"}, "Tier 1+fence failed"
assert safe_parse_json(samples["in_prose"]) == {"a": 1, "b": "hello"}, "Tier 2 failed"
g = safe_parse_json(samples["garbage"])
assert g.get("parse_error") is True, "Tier 3 failed"
print("\nAll three tiers behave as expected.")


---

## Part 9: End-to-End Run on All RFPs

Run `process_rfp` against the baseline plus every synthetic fixture. Expected
behaviors flagged inline:

- `adversarial` -- validator should raise >=1 contradiction-style flag.
- `edge_cases` -- retriever should mark >=1 question as `kb-gap`.


In [ ]:
all_runs = {}

print("Running baseline...")
all_runs["baseline"] = baseline_result  # reuse from Part 6

for name, rfp_text in SYNTHETIC_RFPS.items():
    print(f"Running {name}...")
    all_runs[name] = process_rfp(rfp_text, max_revisions=1, rfp_name=name)

print()
header = f"{'RFP':<16} {'parsed':>7} {'flagged':>8} {'revised':>8} {'kb_gaps':>8} {'parse_err':>10}"
print(header)
print("-" * len(header))
for name, result in all_runs.items():
    sb = result["scoreboard"]
    print(f"{name:<16} {sb['parsed']:>7} {sb['flagged']:>8} {sb['revised']:>8} "
          f"{sb['kb_gaps']:>8} {sb['parse_errors']:>10}")

# Inline expectation checks (informational, not hard failures).
adv = all_runs.get("adversarial", {})
edge = all_runs.get("edge_cases", {})
print()
print(f"adversarial validator flags: {len(adv.get('validation', {}).get('issues', []))} "
      f"(expected >= 1)")
print(f"edge_cases kb_gaps: {edge.get('scoreboard', {}).get('kb_gaps', 0)} "
      f"(expected >= 1)")


---

## Part 10: HTML Companion -- `helios_agent.html`

The Bounteous-branded `helios_agent.html` in this folder is the **user-facing UI**
for this same agent team. It is a single self-contained file: paste your API key,
paste an RFP, watch the five agents work the question card-by-card in a live trace.

The HTML mirrors the notebook 1:1. **Any change here must also land in the HTML.**

| Python (notebook) | JavaScript (`helios_agent.html`) |
|-------------------|----------------------------------|
| `KNOWLEDGE_BASE`  | `const KNOWLEDGE_BASE = { ... }` |
| `search_knowledge_base()` | `searchKnowledgeBase(query, category)` |
| `SEARCH_KB_TOOL`  | `SEARCH_KB_TOOL` constant |
| `PARSER_PROMPT` / `RETRIEVER_PROMPT` / `DRAFTER_PROMPT` / `VALIDATOR_PROMPT` / `REVISER_PROMPT` | matching JS string constants |
| `run_parser`      | `runParser(rfpText)` |
| `run_retriever`   | `runRetriever(question)` |
| `run_drafter`     | `runDrafter(question, hits, kbGap)` |
| `run_validator`   | `runValidator(drafts)` |
| `run_reviser`     | `runReviser(question, hits, draft, critique)` |
| `safe_parse_json` | `safeParseJson(text)` |
| `process_rfp`     | `processRfp(rfpText, maxRevisions)` |
| `SYNTHETIC_RFPS`  | `SYNTHETIC_RFPS` constant (same fixtures) |

The HTML is **hand-authored**, not auto-generated from this notebook. If you
update a prompt or add a KB entry, update both files in the same commit.


---

## Part 11: Stretch -- Evals

The original Part 8 `run_evals` only checked source presence. We extend it with
assertions that reflect the failure budget:

- Every answer has >=1 source (when not a kb_gap).
- `confidence` is in `{high, medium, low}`.
- Adversarial RFP raises >=1 validator contradiction.
- Edge-case RFP yields >=1 `kb-gap` flag.
- Low-confidence answers do not fabricate specific dollar amounts or year-like dates.


In [ ]:
import re as _re

_NUMERIC_DOLLAR = _re.compile(r"\$\s?\d")
_NUMERIC_YEAR   = _re.compile(r"\b(19|20)\d{2}\b")

def run_evals(rfp_name: str, result: dict) -> dict:
    """Extended evals. Returns {passed, failed, details:[{assertion, passed, note}]}."""
    out = {"rfp": rfp_name, "passed": 0, "failed": 0, "details": []}

    def check(name, ok, note=""):
        out["details"].append({"assertion": name, "passed": bool(ok), "note": note})
        if ok:
            out["passed"] += 1
        else:
            out["failed"] += 1

    answers = result.get("answers", [])

    # 1. Every answer has >=1 source (unless flagged kb-gap).
    for a in answers:
        qid = a.get("question_id", "?")
        is_gap = "kb-gap" in a.get("flags", [])
        check(
            f"{qid} has_sources",
            len(a.get("sources", [])) > 0 or is_gap,
            "kb-gap exempt" if is_gap else "",
        )

    # 2. Confidence in valid set.
    for a in answers:
        qid = a.get("question_id", "?")
        check(f"{qid} confidence_valid",
              a.get("confidence") in {"high", "medium", "low"})

    # 3. Adversarial validator catches >=1 contradiction.
    if rfp_name == "adversarial":
        issues = result.get("validation", {}).get("issues", [])
        has_contra = any(i.get("issue_type") == "contradiction" for i in issues)
        check("adversarial_contradiction_caught", has_contra,
              note=f"{len(issues)} issues total")

    # 4. Edge-case RFP yields >=1 kb-gap.
    if rfp_name == "edge_cases":
        gaps = sum(1 for a in answers if "kb-gap" in a.get("flags", []))
        check("edge_cases_kb_gap_flagged", gaps >= 1, note=f"{gaps} kb-gap answers")

    # 5. Low-confidence answers do not fabricate specifics.
    for a in answers:
        if a.get("confidence") != "low":
            continue
        text = a.get("answer", "")
        bad = bool(_NUMERIC_DOLLAR.search(text)) or bool(_NUMERIC_YEAR.search(text))
        check(f"{a.get('question_id','?')} low_conf_no_fabrication",
              not bad,
              note="found numeric specifics in low-confidence answer" if bad else "")

    return out


# Run evals against every synthetic RFP.
eval_summary = {}
for name, result in all_runs.items():
    e = run_evals(name, result)
    eval_summary[name] = e
    print(f"=== {name}: {e['passed']} passed / {e['failed']} failed ===")
    for d in e["details"]:
        mark = "PASS" if d["passed"] else "FAIL"
        note = f"  ({d['note']})" if d["note"] else ""
        print(f"  [{mark}] {d['assertion']}{note}")
    print()
